# Travel-time-based 30 min E2SFCA accessibility and longitudinal analysis

This notebook reads the already executed Travel-time-based 30 min E2SFCA outputs. It does not recalculate E2SFCA or redefine A/A*.

**Workflow:** E2SFCA Accessibility → fixed A* standardisation → annual HP–LA → longitudinal trajectories → maps and statistical summaries.

- Final geography: fixed 3,411 2021 LSOAs.
- Annual HP–LA: `care50_rate > annual median` and `A* < annual median`, using strict inequalities.
- Trajectories: Persistent, Emerging, Resolved / improved, Intermittent and Never HP–LA.
- No Bi-LISA, regression or BYM2 is run.
- All figures are saved as 300-dpi PNG and vector PDF.

## Reproducibility contract

**Inputs.** The executed exact-halo 30-minute E2SFCA files and fixed 2021 geography. This notebook reads A and A* unchanged.

**Classification.** Annual HP–LA uses strict above-median Care50 and below-median A*. Trajectories are derived from the three annual binary statuses. ICB and urban–rural sections are descriptive decompositions on the fixed geography.

**Outputs.** Summary tables, annual status/trajectory files, maps and QA. Only aggregate summaries and QA are distributed publicly; LSOA-level files remain in the licensed/private data environment.


## 1. Load and audit the executed E2SFCA outputs

In [ ]:
from pathlib import Path
import sys

sys.dont_write_bytecode = True
RUN_DIR = Path.cwd().resolve()
sys.path.insert(0, str(RUN_DIR / "scripts"))
from downstream_workflow import AccessibilityLongitudinalWorkflow

workflow = AccessibilityLongitudinalWorkflow(RUN_DIR)
input_audit = workflow.load_and_audit_inputs()
input_audit

## 2. Load A and A* without recalculating E2SFCA

In [ ]:
accessibility_input_audit = workflow.load_accessibility_outputs()
accessibility_input_audit

## 3. A/A* summary and fixed-benchmark statistics

In [ ]:
accessibility_statistics = workflow.write_accessibility_statistics()
accessibility_statistics

## 4. Three-year A* maps and distributions

In [ ]:
accessibility_map_audit = workflow.make_accessibility_maps()
accessibility_map_audit

## 5. Temporal A/A* change maps and statistics

In [ ]:
accessibility_change_summary = workflow.write_changes()
accessibility_change_summary

## 6. ICB-level accessibility summaries

In [ ]:
icb_accessibility_summary = workflow.write_icb_summary()
icb_accessibility_summary

## 7. Annual median-rule HP–LA classifications and maps

In [ ]:
annual_hpla_summary = workflow.classify_annual_hpla()
annual_hpla_summary

## 8. Status sequences, trajectory classes, maps and ICB composition

In [ ]:
trajectory_summary = workflow.build_trajectories()
trajectory_summary

## 9. HP–LA and four-state transition statistics

In [ ]:
transition_summary = workflow.write_transition_statistics()
transition_summary

## 10. Final QA, method manifest and source-file integrity

In [ ]:
final_qa = workflow.final_qa_and_method()
final_qa

## 11. Appended descriptive geographic decomposition: ICB and existing urban–rural classification

This appended section uses only the frozen LSOA-level annual HP–LA and trajectory outputs above. It does not rerun E2SFCA, redefine A/A*, alter annual medians, or reconstruct trajectories.


In [ ]:
# Appended descriptive decomposition only: load frozen LSOA-level outputs; do not recompute E2SFCA or trajectories.
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPEC_LABEL = 'Travel-time-based 30 min'
RUN_DIR = Path.cwd().resolve()
NOTEBOOK_PATH = RUN_DIR / 'icb_urban_rural_decomposition.ipynb'
OUT_DIR = Path(globals().get("_DECOMP_OUTPUT_OVERRIDE", RUN_DIR / "ICB_UrbanRural_Decomposition"))
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
QA_DIR = OUT_DIR / "qa"

if OUT_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite existing decomposition directory: {OUT_DIR}")
OUT_DIR.mkdir(parents=False, exist_ok=False)
TABLE_DIR.mkdir(exist_ok=False)
FIGURE_DIR.mkdir(exist_ok=False)
QA_DIR.mkdir(exist_ok=False)

ICB_CODE_FIELD = "ICB23CD"
ICB_NAME_FIELD = "ICB23NM"
URBAN_RURAL_FIELD = "rural_binary"
URBAN_RURAL_LABELS = {0: "Urban", 1: "Rural"}
YEARS = [2001, 2011, 2021]
TRAJECTORY_ORDER = ['Persistent HP–LA', 'Emerging HP–LA', 'Resolved / improved', 'Intermittent', 'Never HP–LA']
NEW_OUTPUTS = []

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_csv_new(frame, path):
    path = Path(path)
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite: {path}")
    frame.to_csv(path, index=False)
    NEW_OUTPUTS.append(path)
    return path

def write_text_new(text, path):
    path = Path(path)
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite: {path}")
    path.write_text(text, encoding="utf-8")
    NEW_OUTPUTS.append(path)
    return path

def save_figure_new(fig, stem):
    paths = []
    for suffix in ("png", "pdf"):
        path = FIGURE_DIR / f"{stem}.{suffix}"
        if path.exists():
            raise FileExistsError(f"Refusing to overwrite: {path}")
        fig.savefig(path, dpi=400 if suffix == "png" else None, bbox_inches="tight", facecolor="white")
        NEW_OUTPUTS.append(path)
        paths.append(path)
    plt.close(fig)
    return paths

def weighted_mean(values, weights):
    valid = values.notna() & weights.notna()
    denominator = weights.loc[valid].sum()
    return np.nan if denominator <= 0 else np.average(values.loc[valid], weights=weights.loc[valid])

# Hash every pre-existing file except the notebook; the required new directory is not yet present.
PREEXISTING_HASHES = {
    str(path.relative_to(RUN_DIR)): file_sha256(path)
    for path in RUN_DIR.rglob("*")
    if path.is_file() and path != NOTEBOOK_PATH and "ICB_UrbanRural_Decomposition" not in path.parts
}

annual_frames = []
annual_sources = []
for year in YEARS:
    source = RUN_DIR / "tables" / f"annual_hp_la_status_{year}.csv"
    annual_sources.append(source)
    frame = pd.read_csv(source)
    required = {
        "year", "lsoa_code", ICB_CODE_FIELD, ICB_NAME_FIELD, URBAN_RURAL_FIELD,
        "care50_num", "care50_rate", "accessibility_Astar", "within_year_relative_hpla"
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise KeyError(f"{source} missing required fields: {missing}")
    if len(frame) != 3411 or frame["lsoa_code"].nunique() != 3411:
        raise ValueError(f"{source} is not the expected 3,411 unique-LSOA output")
    annual_frames.append(frame)

annual = pd.concat(annual_frames, ignore_index=True)
annual[URBAN_RURAL_FIELD] = pd.to_numeric(annual[URBAN_RURAL_FIELD], errors="raise").astype(int)
if set(annual[URBAN_RURAL_FIELD].unique()) != {0, 1} or annual[URBAN_RURAL_FIELD].isna().any():
    raise ValueError("Existing rural_binary field is not a complete 0/1 classification")
classification_consistency = annual.groupby("lsoa_code")[URBAN_RURAL_FIELD].nunique(dropna=False)
if not classification_consistency.eq(1).all():
    raise ValueError("rural_binary is not constant across the three common-2021 annual outputs")
annual["settlement_type"] = annual[URBAN_RURAL_FIELD].map(URBAN_RURAL_LABELS)
annual["within_year_relative_hpla"] = annual["within_year_relative_hpla"].astype(bool)

trajectory_source = RUN_DIR / "tables" / "lsoa_hp_la_trajectories_2001_2021.csv"
trajectory = pd.read_csv(trajectory_source)
if len(trajectory) != 3411 or trajectory["lsoa_code"].nunique() != 3411:
    raise ValueError("Trajectory source is not the expected 3,411 unique-LSOA output")
membership = annual.loc[annual["year"].eq(2021), ["lsoa_code", ICB_CODE_FIELD, ICB_NAME_FIELD, URBAN_RURAL_FIELD, "settlement_type"]]
trajectory = trajectory.drop(columns=[ICB_CODE_FIELD, ICB_NAME_FIELD], errors="ignore").merge(
    membership, on="lsoa_code", how="left", validate="one_to_one"
)
if trajectory[[ICB_CODE_FIELD, ICB_NAME_FIELD, "settlement_type"]].isna().any().any():
    raise ValueError("Incomplete membership after joining existing trajectory and 2021 covariate fields")

icb_lookup = annual[[ICB_CODE_FIELD, ICB_NAME_FIELD]].drop_duplicates().sort_values(ICB_NAME_FIELD)
if len(icb_lookup) != 7:
    raise ValueError(f"Expected seven ICBs; found {len(icb_lookup)}")
icb_names = icb_lookup[ICB_NAME_FIELD].tolist()
short_name_map = {
    "Bath and North East Somerset, Swindon and Wiltshire": "BSW",
    "Bristol, North Somerset and South Gloucestershire": "BNSSG",
    "Cornwall and the Isles of Scilly": "Cornwall & IoS",
    "Devon": "Devon",
    "Dorset": "Dorset",
    "Gloucestershire": "Gloucestershire",
    "Somerset": "Somerset",
}

print(f"Specification: {SPEC_LABEL}")
print(f"Frozen annual sources: {len(annual_sources)}; frozen trajectory source: {trajectory_source.name}")
print(f"ICB field: {ICB_CODE_FIELD} / {ICB_NAME_FIELD}; ICBs found: {len(icb_lookup)}")
print(f"Existing urban/rural field: {URBAN_RURAL_FIELD}; categories: {URBAN_RURAL_LABELS}")
print("Spatial basis: the same rural_binary classification defined on the harmonised 2021 LSOA geography is held fixed for 2001, 2011 and 2021.")
print("No E2SFCA, A*, HP-LA threshold, or trajectory definition was recalculated or redefined.")

### 11.1 ICB decomposition

Seven-ICB summaries, A* change, and trajectory composition.


In [ ]:
def annual_summary_by(group_fields):
    rows = []
    for keys, group in annual.groupby(group_fields, dropna=False, observed=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_fields, keys))
        row.update({
            "n_lsoas": len(group),
            "mean_care50_rate": group["care50_rate"].mean(),
            "median_care50_rate": group["care50_rate"].median(),
            "mean_Astar": group["accessibility_Astar"].mean(),
            "median_Astar": group["accessibility_Astar"].median(),
            "care50_weighted_mean_Astar": weighted_mean(group["accessibility_Astar"], group["care50_num"]),
            "annual_relative_hpla_count": int(group["within_year_relative_hpla"].sum()),
            "annual_relative_hpla_percent": 100 * group["within_year_relative_hpla"].mean(),
        })
        rows.append(row)
    return pd.DataFrame(rows)

icb_summary = annual_summary_by([ICB_CODE_FIELD, ICB_NAME_FIELD, "year"]).sort_values([ICB_NAME_FIELD, "year"])
write_csv_new(icb_summary, TABLE_DIR / "icb_summary.csv")

icb_changes = []
for start, end in [(2001, 2011), (2011, 2021), (2001, 2021)]:
    left = icb_summary.loc[icb_summary["year"].eq(start), [ICB_CODE_FIELD, ICB_NAME_FIELD, "mean_Astar", "median_Astar"]]
    right = icb_summary.loc[icb_summary["year"].eq(end), [ICB_CODE_FIELD, ICB_NAME_FIELD, "mean_Astar", "median_Astar"]]
    joined = left.merge(right, on=[ICB_CODE_FIELD, ICB_NAME_FIELD], suffixes=(f"_{start}", f"_{end}"), validate="one_to_one")
    joined["period"] = f"{start}-{end}"
    joined["mean_Astar_change"] = joined[f"mean_Astar_{end}"] - joined[f"mean_Astar_{start}"]
    joined["median_Astar_change"] = joined[f"median_Astar_{end}"] - joined[f"median_Astar_{start}"]
    icb_changes.append(joined[[ICB_CODE_FIELD, ICB_NAME_FIELD, "period", "mean_Astar_change", "median_Astar_change"]])
icb_change_summary = pd.concat(icb_changes, ignore_index=True)
write_csv_new(icb_change_summary, TABLE_DIR / "icb_astar_change_summary.csv")

icb_trajectory = (
    trajectory.groupby([ICB_CODE_FIELD, ICB_NAME_FIELD, "trajectory_category"], observed=True)
    .size().rename("trajectory_count").reset_index()
)
full_icb_traj = pd.MultiIndex.from_product(
    [icb_lookup[ICB_CODE_FIELD], TRAJECTORY_ORDER], names=[ICB_CODE_FIELD, "trajectory_category"]
).to_frame(index=False).merge(icb_lookup, on=ICB_CODE_FIELD, how="left")
icb_trajectory = full_icb_traj.merge(
    icb_trajectory, on=[ICB_CODE_FIELD, ICB_NAME_FIELD, "trajectory_category"], how="left"
)
icb_trajectory["trajectory_count"] = icb_trajectory["trajectory_count"].fillna(0).astype(int)
icb_trajectory["icb_lsoas"] = icb_trajectory.groupby(ICB_CODE_FIELD)["trajectory_count"].transform("sum")
icb_trajectory["trajectory_percent"] = 100 * icb_trajectory["trajectory_count"] / icb_trajectory["icb_lsoas"]
icb_trajectory["trajectory_category"] = pd.Categorical(icb_trajectory["trajectory_category"], TRAJECTORY_ORDER, ordered=True)
icb_trajectory = icb_trajectory.sort_values([ICB_NAME_FIELD, "trajectory_category"])
write_csv_new(icb_trajectory, TABLE_DIR / "icb_trajectory_composition.csv")

endpoint_labels = {
    "Bath and North East Somerset, Swindon and Wiltshire": "Bath and North East Somerset,\nSwindon and Wiltshire",
    "Bristol, North Somerset and South Gloucestershire": "Bristol, North Somerset\nand South Gloucestershire",
    "Cornwall and the Isles of Scilly": "Cornwall and the Isles of Scilly",
    "Devon": "Devon",
    "Dorset": "Dorset",
    "Gloucestershire": "Gloucestershire",
    "Somerset": "Somerset",
}
endpoint_label_y = {
    "Bath and North East Somerset, Swindon and Wiltshire": 1.385,
    "Bristol, North Somerset and South Gloucestershire": 1.455,
    "Cornwall and the Isles of Scilly": 0.976,
    "Devon": 1.145,
    "Dorset": 1.195,
    "Gloucestershire": 1.488,
    "Somerset": 1.250,
}

fig, ax = plt.subplots(figsize=(10.3, 6.8))
for name, group in icb_summary.groupby(ICB_NAME_FIELD, sort=True):
    group = group.sort_values("year")
    line, = ax.plot(group["year"], group["mean_Astar"], marker="o", markersize=6, linewidth=2)
    ax.text(2022.0, endpoint_label_y[name], endpoint_labels[name], color=line.get_color(), fontsize=10.5, va="center", linespacing=0.95)

ax.hlines(1.0, xmin=1999, xmax=2021, linestyle="--", linewidth=1.0, color="0.25", zorder=0)
ax.set_xlabel("Census year", fontsize=12)
ax.set_ylabel("Mean relative potential accessibility (A*)", fontsize=12)
ax.set_xticks(YEARS)
ax.set_xlim(1999, 2033)
ax.set_ylim(0.68, 1.58)
ax.tick_params(axis="both", labelsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.spines["bottom"].set_bounds(1999, 2021)
fig.subplots_adjust(left=0.12, right=0.98, bottom=0.14, top=0.97)
save_figure_new(fig, "icb_astar_over_time_direct_labels")

trajectory_plot = icb_trajectory.pivot(index=ICB_NAME_FIELD, columns="trajectory_category", values="trajectory_percent").reindex(columns=TRAJECTORY_ORDER)
trajectory_plot.index = [short_name_map.get(name, name) for name in trajectory_plot.index]
icb_plot_order = trajectory_plot["Persistent HP–LA"].sort_values(ascending=False).index.tolist()
trajectory_plot = trajectory_plot.reindex(icb_plot_order)
trajectory_colours = {
    "Persistent HP–LA": "#b2182b",
    "Emerging HP–LA": "#ef8a62",
    "Resolved / improved": "#4daf4a",
    "Intermittent": "#6a51a3",
    "Never HP–LA": "#d9d9d9",
}
with plt.rc_context({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}):
    fig, ax = plt.subplots(figsize=(10.2, 5.6))
    trajectory_plot.plot(
        kind="bar", stacked=True, ax=ax, width=.585,
        color=[trajectory_colours[category] for category in TRAJECTORY_ORDER],
    )
    ax.set_axisbelow(True)
    ax.set_ylim(0, 105)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.yaxis.grid(True, color="#d9d9d9", linewidth=0.6, alpha=0.75)
    ax.xaxis.grid(False)
    ax.set(xlabel="ICB", ylabel="Percentage of LSOAs", title=f"HP–LA trajectory composition by ICB: {SPEC_LABEL}")
    ax.legend(title="Trajectory", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8)
    ax.tick_params(axis="x", rotation=25)
    save_figure_new(fig, "icb_trajectory_percentages")

print("Compact ICB summary (2021):")
print(icb_summary.loc[icb_summary["year"].eq(2021), [ICB_NAME_FIELD, "n_lsoas", "mean_Astar", "median_Astar", "annual_relative_hpla_count", "annual_relative_hpla_percent"]].round(3).to_string(index=False))

### 11.2 Overall urban–rural comparison

Uses the existing `rural_binary` field defined on the harmonised 2021 LSOA geography (0 = Urban, 1 = Rural), held fixed across all three years; no year-specific classification is inferred or downloaded.


In [ ]:
urban_rural_summary = annual_summary_by(["year", "settlement_type"]).sort_values(["year", "settlement_type"])
write_csv_new(urban_rural_summary, TABLE_DIR / "urban_rural_overall_summary.csv")

urban_rural_contrast = []
for year, group in urban_rural_summary.groupby("year"):
    indexed = group.set_index("settlement_type")
    urban_rural_contrast.append({
        "year": year,
        "urban_n": int(indexed.loc["Urban", "n_lsoas"]) if "Urban" in indexed.index else 0,
        "rural_n": int(indexed.loc["Rural", "n_lsoas"]) if "Rural" in indexed.index else 0,
        "rural_minus_urban_mean_Astar": indexed.loc["Rural", "mean_Astar"] - indexed.loc["Urban", "mean_Astar"],
        "rural_minus_urban_median_Astar": indexed.loc["Rural", "median_Astar"] - indexed.loc["Urban", "median_Astar"],
    })
urban_rural_contrast = pd.DataFrame(urban_rural_contrast)
write_csv_new(urban_rural_contrast, TABLE_DIR / "urban_rural_astar_contrast.csv")

urban_rural_trajectory = (
    trajectory.groupby(["settlement_type", "trajectory_category"], observed=True)
    .size().rename("trajectory_count").reset_index()
)
full_ur_traj = pd.MultiIndex.from_product(
    [["Urban", "Rural"], TRAJECTORY_ORDER], names=["settlement_type", "trajectory_category"]
).to_frame(index=False)
urban_rural_trajectory = full_ur_traj.merge(urban_rural_trajectory, how="left", on=["settlement_type", "trajectory_category"])
urban_rural_trajectory["trajectory_count"] = urban_rural_trajectory["trajectory_count"].fillna(0).astype(int)
urban_rural_trajectory["group_lsoas"] = urban_rural_trajectory.groupby("settlement_type")["trajectory_count"].transform("sum")
urban_rural_trajectory["trajectory_percent"] = 100 * urban_rural_trajectory["trajectory_count"] / urban_rural_trajectory["group_lsoas"]
write_csv_new(urban_rural_trajectory, TABLE_DIR / "urban_rural_trajectory_composition.csv")

series_colours = {"Rural": "#1f77b4", "Urban": "#ff7f0e"}
with plt.rc_context({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 10.0,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}):
    plot_specs = [
        (
            "urban_rural_mean_accessibility",
            "mean_Astar",
            "Mean A*",
            f"Urban–rural mean accessibility: {SPEC_LABEL}",
        ),
        (
            "urban_rural_hpla_prevalence",
            "annual_relative_hpla_percent",
            "Annual relative HP–LA (%)",
            f"Urban–rural HP–LA prevalence: {SPEC_LABEL}",
        ),
    ]

    # The former 7.2-inch two-panel width is reduced by 25% to 5.4 inches.
    for stem, value_field, y_label, title in plot_specs:
        fig, ax = plt.subplots(figsize=(5.4, 4.8), constrained_layout=True)
        fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.04)
        for settlement in ["Rural", "Urban"]:
            group = urban_rural_summary.loc[
                urban_rural_summary["settlement_type"].eq(settlement)
            ].sort_values("year")
            colour = series_colours[settlement]
            ax.plot(
                group["year"], group[value_field],
                color=colour, linewidth=2.4, marker="o", markersize=6.0,
                markerfacecolor=colour, markeredgecolor=colour, label=settlement,
            )

        ax.set(xlabel="Year", ylabel=y_label, title=title)
        ax.set_xticks(YEARS)
        ax.grid(axis="y", alpha=.25)
        ax.legend(frameon=False)
        save_figure_new(fig, stem)

print("Overall urban–rural summary:")
print(urban_rural_summary[["year", "settlement_type", "n_lsoas", "mean_care50_rate", "median_care50_rate", "mean_Astar", "median_Astar", "annual_relative_hpla_count", "annual_relative_hpla_percent"]].round(3).to_string(index=False))
print("\nRural minus urban A* contrast:")
print(urban_rural_contrast.round(3).to_string(index=False))

### 11.3 ICB × urban–rural decomposition

All subgroup sample sizes are printed before interpretation; zero-observation groups would be retained with unavailable metrics.


In [ ]:
# Explicit subgroup sample sizes before interpretation; no small group is suppressed.
subgroup_sizes = (
    membership.groupby([ICB_CODE_FIELD, ICB_NAME_FIELD, "settlement_type"], observed=True)
    .size().rename("n_lsoas").reset_index()
)
full_subgroups = pd.MultiIndex.from_product(
    [icb_lookup[ICB_CODE_FIELD], ["Urban", "Rural"]], names=[ICB_CODE_FIELD, "settlement_type"]
).to_frame(index=False).merge(icb_lookup, on=ICB_CODE_FIELD, how="left")
subgroup_sizes = full_subgroups.merge(subgroup_sizes, on=[ICB_CODE_FIELD, ICB_NAME_FIELD, "settlement_type"], how="left")
subgroup_sizes["n_lsoas"] = subgroup_sizes["n_lsoas"].fillna(0).astype(int)
write_csv_new(subgroup_sizes, TABLE_DIR / "icb_x_urban_rural_sample_sizes.csv")
print("ICB × urban/rural subgroup sample sizes (printed before interpretation; zero groups retained):")
print(subgroup_sizes.sort_values([ICB_NAME_FIELD, "settlement_type"]).to_string(index=False))

observed_icb_ur = annual_summary_by([ICB_CODE_FIELD, ICB_NAME_FIELD, "year", "settlement_type"])
full_index = pd.MultiIndex.from_product(
    [icb_lookup[ICB_CODE_FIELD], YEARS, ["Urban", "Rural"]],
    names=[ICB_CODE_FIELD, "year", "settlement_type"],
).to_frame(index=False).merge(icb_lookup, on=ICB_CODE_FIELD, how="left")
icb_x_urban_rural = full_index.merge(
    observed_icb_ur,
    on=[ICB_CODE_FIELD, ICB_NAME_FIELD, "year", "settlement_type"],
    how="left",
)
icb_x_urban_rural["n_lsoas"] = icb_x_urban_rural["n_lsoas"].fillna(0).astype(int)
icb_x_urban_rural = icb_x_urban_rural.sort_values([ICB_NAME_FIELD, "year", "settlement_type"])
write_csv_new(icb_x_urban_rural, TABLE_DIR / "icb_x_urban_rural_summary.csv")

compact_rows = []
for (code, name, year), group in icb_x_urban_rural.groupby([ICB_CODE_FIELD, ICB_NAME_FIELD, "year"], observed=True):
    indexed = group.set_index("settlement_type")
    row = {ICB_CODE_FIELD: code, ICB_NAME_FIELD: name, "year": year}
    for settlement in ["Urban", "Rural"]:
        lower = settlement.lower()
        row[f"{lower}_n"] = int(indexed.loc[settlement, "n_lsoas"]) if settlement in indexed.index else 0
        row[f"{lower}_mean_Astar"] = indexed.loc[settlement, "mean_Astar"] if settlement in indexed.index else np.nan
        row[f"{lower}_hpla_percent"] = indexed.loc[settlement, "annual_relative_hpla_percent"] if settlement in indexed.index else np.nan
    row["rural_minus_urban_mean_Astar"] = row["rural_mean_Astar"] - row["urban_mean_Astar"]
    row["rural_minus_urban_hpla_percentage_points"] = row["rural_hpla_percent"] - row["urban_hpla_percent"]
    compact_rows.append(row)
icb_x_compact = pd.DataFrame(compact_rows).sort_values([ICB_NAME_FIELD, "year"])
write_csv_new(icb_x_compact, TABLE_DIR / "icb_x_urban_rural_compact.csv")

icb_x_trajectory = (
    trajectory.groupby([ICB_CODE_FIELD, ICB_NAME_FIELD, "settlement_type", "trajectory_category"], observed=True)
    .size().rename("trajectory_count").reset_index()
)
full_icb_ur_traj = pd.MultiIndex.from_product(
    [icb_lookup[ICB_CODE_FIELD], ["Urban", "Rural"], TRAJECTORY_ORDER],
    names=[ICB_CODE_FIELD, "settlement_type", "trajectory_category"],
).to_frame(index=False).merge(icb_lookup, on=ICB_CODE_FIELD, how="left")
icb_x_trajectory = full_icb_ur_traj.merge(
    icb_x_trajectory,
    on=[ICB_CODE_FIELD, ICB_NAME_FIELD, "settlement_type", "trajectory_category"],
    how="left",
)
icb_x_trajectory["trajectory_count"] = icb_x_trajectory["trajectory_count"].fillna(0).astype(int)
group_n = subgroup_sizes.rename(columns={"n_lsoas": "subgroup_lsoas"})
icb_x_trajectory = icb_x_trajectory.merge(group_n, on=[ICB_CODE_FIELD, ICB_NAME_FIELD, "settlement_type"], how="left")
icb_x_trajectory["trajectory_percent"] = np.where(
    icb_x_trajectory["subgroup_lsoas"].gt(0),
    100 * icb_x_trajectory["trajectory_count"] / icb_x_trajectory["subgroup_lsoas"],
    np.nan,
)
write_csv_new(icb_x_trajectory, TABLE_DIR / "icb_x_urban_rural_trajectory_composition.csv")

fig, axes = plt.subplots(1, 3, figsize=(15.0, 6.0), sharey=True)
for ax, year in zip(axes, YEARS):
    subset = icb_x_compact.loc[icb_x_compact["year"].eq(year)].sort_values(ICB_NAME_FIELD)
    y = np.arange(len(subset))
    ax.plot(subset["urban_mean_Astar"], y, "o", label="Urban")
    ax.plot(subset["rural_mean_Astar"], y, "s", label="Rural")
    for yy, urban_value, rural_value in zip(y, subset["urban_mean_Astar"], subset["rural_mean_Astar"]):
        if pd.notna(urban_value) and pd.notna(rural_value):
            ax.plot([urban_value, rural_value], [yy, yy], color="0.75", zorder=0)
    ax.set_title(str(year)); ax.set_xlabel("Mean A*"); ax.grid(axis="x", alpha=.2)
    ax.set_yticks(y, [short_name_map.get(name, name) for name in subset[ICB_NAME_FIELD]])
axes[0].set_ylabel("ICB")
axes[-1].legend(frameon=False)
fig.suptitle(f"Urban–rural A* contrast within ICBs: {SPEC_LABEL}")
save_figure_new(fig, "icb_x_urban_rural_astar_contrast")

fig, axes = plt.subplots(1, 3, figsize=(15.0, 6.0), sharey=True)
for ax, year in zip(axes, YEARS):
    subset = icb_x_compact.loc[icb_x_compact["year"].eq(year)].sort_values(ICB_NAME_FIELD)
    y = np.arange(len(subset))
    ax.plot(subset["urban_hpla_percent"], y, "o", label="Urban")
    ax.plot(subset["rural_hpla_percent"], y, "s", label="Rural")
    for yy, urban_value, rural_value in zip(y, subset["urban_hpla_percent"], subset["rural_hpla_percent"]):
        if pd.notna(urban_value) and pd.notna(rural_value):
            ax.plot([urban_value, rural_value], [yy, yy], color="0.75", zorder=0)
    ax.set_title(str(year)); ax.set_xlabel("Annual relative HP–LA (%)"); ax.grid(axis="x", alpha=.2)
    ax.set_yticks(y, [short_name_map.get(name, name) for name in subset[ICB_NAME_FIELD]])
axes[0].set_ylabel("ICB")
axes[-1].legend(frameon=False)
fig.suptitle(f"Urban–rural HP–LA contrast within ICBs: {SPEC_LABEL}")
save_figure_new(fig, "icb_x_urban_rural_hpla_prevalence")

print("\nCompact ICB × urban/rural comparison:")
print(icb_x_compact.round(3).to_string(index=False))

### 11.4 Descriptive interpretation, preservation QA, and manifest

Interpretation is descriptive only and avoids causal claims.


In [ ]:
summary_lines = [
    f"# Descriptive interpretation summary: {SPEC_LABEL}",
    "",
    "All statements below are descriptive associations based on the frozen annual HP–LA and trajectory outputs.",
    "",
]
highest_names, lowest_names, hpla_names, lower_settlements = [], [], [], []
for year in YEARS:
    year_icb = icb_summary.loc[icb_summary["year"].eq(year)]
    highest = year_icb.loc[year_icb["mean_Astar"].idxmax()]
    lowest = year_icb.loc[year_icb["mean_Astar"].idxmin()]
    highest_hpla = year_icb.loc[year_icb["annual_relative_hpla_percent"].idxmax()]
    year_ur = urban_rural_summary.loc[urban_rural_summary["year"].eq(year)].set_index("settlement_type")
    lower_group = year_ur["mean_Astar"].idxmin()
    highest_names.append(highest[ICB_NAME_FIELD]); lowest_names.append(lowest[ICB_NAME_FIELD])
    hpla_names.append(highest_hpla[ICB_NAME_FIELD]); lower_settlements.append(lower_group)
    summary_lines.extend([
        f"## {year}",
        f"- Highest mean A*: {highest[ICB_NAME_FIELD]} ({highest['mean_Astar']:.3f}); lowest: {lowest[ICB_NAME_FIELD]} ({lowest['mean_Astar']:.3f}).",
        f"- Highest annual relative HP–LA prevalence: {highest_hpla[ICB_NAME_FIELD]} ({highest_hpla['annual_relative_hpla_percent']:.1f}%).",
        f"- {lower_group} LSOAs had the lower overall mean A* ({year_ur.loc[lower_group, 'mean_Astar']:.3f}).",
    ])

summary_lines.extend(["", "## Within-ICB urban–rural composition (2021)"])
latest = icb_x_urban_rural.loc[icb_x_urban_rural["year"].eq(2021)]
for name, group in latest.groupby(ICB_NAME_FIELD, sort=True):
    indexed = group.set_index("settlement_type")
    urban_count = indexed.loc["Urban", "annual_relative_hpla_count"]
    rural_count = indexed.loc["Rural", "annual_relative_hpla_count"]
    urban_prev = indexed.loc["Urban", "annual_relative_hpla_percent"]
    rural_prev = indexed.loc["Rural", "annual_relative_hpla_percent"]
    count_group = "Urban" if urban_count > rural_count else "Rural" if rural_count > urban_count else "Neither"
    prevalence_group = "Urban" if urban_prev > rural_prev else "Rural" if rural_prev > urban_prev else "Neither"
    if count_group == prevalence_group and count_group in {"Urban", "Rural"}:
        composition = f"primarily observed among {count_group.lower()} LSOAs by both count and subgroup prevalence"
    else:
        composition = "mixed: the larger HP–LA count and higher subgroup prevalence were not in the same settlement category"
    summary_lines.append(
        f"- {name}: urban HP–LA {int(urban_count)} ({urban_prev:.1f}%), rural HP–LA {int(rural_count)} ({rural_prev:.1f}%); {composition}."
    )

summary_lines.extend([
    "",
    "## Pattern consistency across years",
    f"- The highest-mean-A* ICB was {'the same in all three years' if len(set(highest_names)) == 1 else 'not the same in all three years'}.",
    f"- The lowest-mean-A* ICB was {'the same in all three years' if len(set(lowest_names)) == 1 else 'not the same in all three years'}.",
    f"- The ICB with the highest annual relative HP–LA prevalence was {'the same in all three years' if len(set(hpla_names)) == 1 else 'not the same in all three years'}.",
    f"- The settlement category with lower overall mean A* was {'consistent across all three years (' + lower_settlements[0] + ')' if len(set(lower_settlements)) == 1 else 'not consistent across all three years'}.",
    "- These comparisons do not establish causal effects and do not alter any analytical classification.",
    "",
])
interpretation_path = write_text_new("\n".join(summary_lines), OUT_DIR / "descriptive_interpretation_summary.md")
print("\n".join(summary_lines))

# QA: source hashes, row conservation, exhaustive trajectory composition, and preservation of all pre-existing non-notebook files.
current_preexisting = {
    str(path.relative_to(RUN_DIR)): file_sha256(path)
    for path in RUN_DIR.rglob("*")
    if path.is_file() and path != NOTEBOOK_PATH and "ICB_UrbanRural_Decomposition" not in path.parts
}
source_paths = annual_sources + [trajectory_source]
qa_checks = [
    ("three annual frozen LSOA sources loaded", len(annual_sources) == 3),
    ("3,411 unique LSOAs in every year", all(len(frame) == 3411 and frame["lsoa_code"].nunique() == 3411 for frame in annual_frames)),
    ("seven existing ICBs found", len(icb_lookup) == 7),
    ("existing rural_binary complete and binary", set(annual[URBAN_RURAL_FIELD].unique()) == {0, 1} and not annual[URBAN_RURAL_FIELD].isna().any()),
    ("harmonised-2021 rural_binary identical across all three years", classification_consistency.eq(1).all()),
    ("all ICB x urban/rural subgroup sizes printed and retained", len(subgroup_sizes) == 14),
    ("ICB annual summaries conserve 3,411 LSOAs", icb_summary.groupby("year")["n_lsoas"].sum().eq(3411).all()),
    ("urban/rural annual summaries conserve 3,411 LSOAs", urban_rural_summary.groupby("year")["n_lsoas"].sum().eq(3411).all()),
    ("ICB x urban/rural annual summaries conserve 3,411 LSOAs", icb_x_urban_rural.groupby("year")["n_lsoas"].sum().eq(3411).all()),
    ("ICB trajectory counts conserve 3,411 LSOAs", int(icb_trajectory["trajectory_count"].sum()) == 3411),
    ("ICB x urban/rural trajectory counts conserve 3,411 LSOAs", int(icb_x_trajectory["trajectory_count"].sum()) == 3411),
    ("all pre-existing non-notebook files unchanged", current_preexisting == PREEXISTING_HASHES),
]
qa_frame = pd.DataFrame(qa_checks, columns=["check", "pass"])
write_csv_new(qa_frame, QA_DIR / "decomposition_qa.csv")
if not qa_frame["pass"].all():
    raise RuntimeError(qa_frame.loc[~qa_frame["pass"]].to_string(index=False))

manifest_path = OUT_DIR / "decomposition_manifest.json"
if manifest_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {manifest_path}")
expected_new_csvs = sorted(str(path) for path in TABLE_DIR.glob("*.csv")) + [str(QA_DIR / "decomposition_qa.csv")]
expected_new_figures = sorted(str(path) for path in FIGURE_DIR.glob("*"))
manifest = {
    "specification": SPEC_LABEL,
    "notebook_modified": str(NOTEBOOK_PATH),
    "cells_appended": 10,
    "frozen_data_files_used": [str(path) for path in source_paths],
    "frozen_data_sha256": {str(path): file_sha256(path) for path in source_paths},
    "icb_fields": [ICB_CODE_FIELD, ICB_NAME_FIELD],
    "urban_rural_field": URBAN_RURAL_FIELD,
    "urban_rural_mapping": {"0": "Urban", "1": "Rural"},
    "urban_rural_spatial_basis": "rural_binary defined on harmonised 2021 LSOA geography and held constant across 2001, 2011 and 2021",
    "number_of_icbs": int(len(icb_lookup)),
    "urban_rural_categories_found": sorted(annual["settlement_type"].unique().tolist()),
    "new_csv_paths": expected_new_csvs,
    "new_figure_paths": expected_new_figures,
    "pre_existing_files_overwritten": False,
    "existing_analytical_definition_changed": False,
    "e2sfca_recomputed": False,
    "trajectory_recomputed_under_new_specification": False,
    "logic_version": "identical descriptive decomposition v1; specification-specific frozen Astar and HP-LA results retained",
    "python": platform.python_version(),
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
NEW_OUTPUTS.append(manifest_path)

output_manifest_path = OUT_DIR / "output_manifest.csv"
if output_manifest_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {output_manifest_path}")
manifest_rows = []
for path in sorted(path for path in OUT_DIR.rglob("*") if path.is_file() and path != output_manifest_path):
    manifest_rows.append({
        "relative_path": path.relative_to(OUT_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    })
pd.DataFrame(manifest_rows).to_csv(output_manifest_path, index=False)
NEW_OUTPUTS.append(output_manifest_path)

print(f"\nManifest complete: {manifest_path}")
print(f"Cells appended: 10; ICBs: {len(icb_lookup)}; categories: {sorted(annual['settlement_type'].unique())}")
print(f"New CSV/manifest files: {len(list(OUT_DIR.rglob('*.csv'))) + len(list(OUT_DIR.rglob('*.json')))}")
print(f"New figure files: {len(list(FIGURE_DIR.glob('*')))}")
print("No pre-existing file was overwritten; no E2SFCA or trajectory definition was recomputed.")